## 1. Instalar y comprobar la versión

Puedes ejecutar este notebook sin los anteriores.


In [ ]:
!pip install --quiet pydantic-ai nest_asyncio httpx

💡 **Nota sobre el mensaje en rojo de arriba:**

Si ves un aviso de pip que habla sobre "dependency conflicts" con paquetes como google-auth u opentelemetry, ¡no te preocupes en absoluto!

Google Colab viene con muchas librerías preinstaladas para interactuar con -Google Cloud. Al instalar pydantic-ai, algunas versiones se actualizan y Colab nos avisa de esa pequeña discrepancia. Para lo que vamos a construir aquí (nuestro agente Don Plátano y sus cotizaciones con Pydantic), todo funcionará de forma impecable. Puedes ignorar el aviso rojo y continuar con total tranquilidad.

In [ ]:
from importlib.metadata import version
print("pydantic-ai:", version("pydantic-ai"))


## 2. Leer OpenRouter desde Secrets

Crea `OPENROUTER_API_KEY` en el panel de Secrets y habilita el acceso desde el notebook.


In [ ]:
import os
import nest_asyncio
from google.colab import userdata

clave = userdata.get("OPENROUTER_API_KEY")
if not clave:
    raise RuntimeError("Falta OPENROUTER_API_KEY en Secrets")
os.environ["OPENROUTER_API_KEY"] = clave
nest_asyncio.apply()
print("✅ Configuración lista")


## 3. Crear el agente y la herramienta de precio

`@agent.tool_plain` registra una función normal. El precio es una convención inventada.


In [ ]:
from pydantic_ai import Agent, RunContext

agente = Agent(
    "openrouter:openai/gpt-4o-mini",
    system_prompt=(
        "Eres Don Plátano, corredor del Banano Exchange. "
        "Responde en español, sin inventar precios ni tasas. "
        "Usa las herramientas para consultar el precio de banana, USD/MXN y calcular. "
        "Indica siempre que el precio de banana es ficticio y cita la fecha de la tasa."
    ),
)

@agente.tool_plain
def precio_banana_mxn() -> float:
    """Devuelve el precio fijo y ficticio de una banana en MXN."""
    return 5.0


## 4. Tool de tipo de cambio

Frankfurter v2 publica tasas de referencia con fecha, no cotizaciones negociables en tiempo real.


In [ ]:
import httpx

@agente.tool_plain
def tipo_cambio_usd_mxn() -> dict:
    """Obtiene tasa de referencia reciente de USD a MXN y fecha de publicación."""
    try:
        r = httpx.get("https://api.frankfurter.dev/v2/rate/USD/MXN", timeout=10.0)
        r.raise_for_status()
        d = r.json()
        tasa = float(d["rate"])
        if tasa <= 0:
            raise ValueError("Tasa no positiva")
        return {"fecha": d["date"], "mxn_por_usd": tasa, "fuente": "Frankfurter v2"}
    except (httpx.HTTPError, KeyError, TypeError, ValueError) as error:
        raise RuntimeError("Fallo de USD/MXN: no inventar la tasa") from error


## 5. Tool con contexto y calculadora

`@agent.tool` añade `RunContext` como primer parámetro; el artículo 4 lo aprovechará para dependencias de un cliente.


In [ ]:
@agente.tool
def convertir_mxn_a_bananas(ctx: RunContext[None], monto_mxn: float, precio_mxn: float) -> float:
    """Convierte MXN a bananas usando el precio obtenido de la herramienta oficial."""
    if monto_mxn < 0 or precio_mxn <= 0:
        raise ValueError("Monto y precio deben ser válidos")
    return round(monto_mxn / precio_mxn, 4)


## 6. Consultar 100 USD

El modelo debe decidir las llamadas. **No des por hecho** que lo hará correctamente.


In [ ]:
pregunta = (
    "¿Cuántas bananas equivalen a 100 USD? "
    "Utiliza las tres herramientas: precio_banana_mxn, tipo_cambio_usd_mxn "
    "y convertir_mxn_a_bananas. Muestra tasa, fecha y operación."
)
resultado = agente.run_sync(pregunta)
print(resultado.output)
print("Tokens:", resultado.usage())


## 7. Auditar las llamadas

Inspecciona `ToolCallPart` y `ToolReturnPart`. La aserción te avisa si el agente omitió alguna herramienta; no prueba que la respuesta final sea matemáticamente correcta.


In [ ]:
from pydantic_ai.messages import ToolCallPart, ToolReturnPart

llamadas = []
for mensaje in resultado.all_messages():
    for parte in mensaje.parts:
        if isinstance(parte, ToolCallPart):
            llamadas.append(parte.tool_name)
            print("LLAMADA:", parte.tool_name, "args:", parte.args)
        elif isinstance(parte, ToolReturnPart):
            print("RETORNO:", parte.tool_name, "resultado:", str(parte.content)[:250])

esperadas = {"precio_banana_mxn", "tipo_cambio_usd_mxn", "convertir_mxn_a_bananas"}
faltantes = esperadas - set(llamadas)
if faltantes:
    print("⚠️ El modelo omitió herramientas:", sorted(faltantes))
else:
    print("✅ El agente llamó las tres herramientas")


## 8. Comparar con el agente sin herramientas

Esta respuesta puede inventar un precio: no la uses como referencia.


In [ ]:
agente_sin_tools = Agent(
    "openrouter:openai/gpt-4o-mini",
    system_prompt="Eres Don Plátano. Responde en español y cotiza todo en bananas.",
)
comparacion = agente_sin_tools.run_sync("¿Cuántas bananas equivalen a 100 USD?")
print("Sin herramientas:", comparacion.output)
print("Con herramientas:", resultado.output)


## 9. Mini-reto: presupuesto personal

Cotiza **250 USD** y comprueba que el modelo llame las herramientas. Bonus: prueba con **1,000 MXN** y razona por qué no debería necesitar consultar USD/MXN.


In [ ]:
# Tu turno:
# reto = agente.run_sync("Tengo un presupuesto de 250 USD. ¿Cuántas bananas equivalen?")
# print(reto.output)
